In [13]:
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [50]:
def anonymize(data: str) -> list:
    df = pd.read_csv(data)
    nb_lines = df.shape[0]
    
    df["NAME"] = ""
    df["NAME"] = [f"{i:04d}.png" for i in range(nb_lines)]
    
    df.to_csv(data, index=False)
    
    
    for idx, row in df.iterrows():
        print(row["SOURCE"])
        
anonymize("source/csv/sources.csv")

"Z:\260701\260701_LEL2266859_L_2\260701_LEL2266859_L_2_HD\png\moment0ff.png"
"Z:\260701\260701_LEL2266859_R_2\260701_LEL2266859_R_2_HD\png\moment0ff.png"
"Z:\260325\260325_ZEN1418552_L\260325_ZEN1418552_L_HD\png\moment0ff.png"
"Z:\260325\260325_ZEN1418552_R\260325_ZEN1418552_R_HD\png\moment0ff.png"
"Z:\260528\260528_CHG_L_1\260528_CHG_L_1_HD\png\moment0ff.png"
"Z:\260528\260528_CHG_R_1\260528_CHG_R_1_HD\png\moment0ff.png"
"Z:\260528\260528_BAH_L\260528_BAH_L_HD\png\moment0ff.png"
"Z:\260528\260528_BAH_R_1\260528_BAH_R_1_HD\png\moment0ff.png"
"Z:\260528\260528_LG2229224_L_1\260528_LG2229224_L_1_HD\png\moment0ff.png"
"Z:\260528\260528_LG2229224_R_1\260528_LG2229224_R_1_HD\png\moment0ff.png"
"Z:\260521\260521_TIG2260592_L_1\260521_TIG2260592_L_1_HD\png\moment0ff.png"
"Z:\260521\260521_TIG2260592_R_1\260521_TIG2260592_R_1_HD\png\moment0ff.png"
"Z:\260617\260617_APM_L\260617_APM_L_HD\png\moment0ff.png"
"Z:\260617\260617_APM_R\260617_APM_R_HD\png\moment0ff.png"
"Z:\260617\260617_ESM_L\260617

In [51]:
def split_into_fold(data: str, n: int) -> list:
    df = pd.read_csv(data)

    sgkf = StratifiedGroupKFold(
        n_splits=n,
        shuffle=True,
        random_state=42
    )
    
    folds = []

    for train_idx, val_idx in sgkf.split(
            X=df["SOURCE"],
            y=df["SIDE"],
            groups=df["PATIENT"] ):
                
        train = df.iloc[train_idx]
        val = df.iloc[val_idx]
        
        folds.append({"train": train, "valid": val})
        
    return folds

def _create_fold_class(dict_ds: dict, dir: Path, class_name: str):
    
    output = dir / class_name
    df = dict_ds[class_name]
    
    left_path = output / "left"
    right_path = output / "right"
    
    left_path.mkdir(parents=True, exist_ok=True)
    right_path.mkdir(parents=True, exist_ok=True)
        
    left_df: pd.DataFrame = df.loc[df["SIDE"] == "L"]
    right_df: pd.DataFrame = df.loc[df["SIDE"] == "R"]
    
    for _, row in left_df.iterrows():
        
        source = row["SOURCE"]
        name = row["NAME"]
        
        source = source.strip('"')
        name = name.strip()

        shutil.copy(source, left_path / name)
    
    for _, row in right_df.iterrows():
        
        source = row["SOURCE"]
        name = row["NAME"]
        
        source = source.strip('"')
        name = name.strip()

        shutil.copy(source, right_path / name)
        
def create_folds(lst_split: list, output_folder: str):
    
    folds = []
    
    output_path = Path(output_folder)
    
    for n_fold, dict_fold in enumerate(lst_split):
        
        fold_path = output_path / f"fold_{n_fold}"
                        
        # create for train
        _create_fold_class(dict_fold, fold_path, "train")
        
        # create for valid
        _create_fold_class(dict_fold, fold_path, "valid")
    
    return folds
        
list_folds = split_into_fold("source/csv/sources.csv", 5)
create_folds(list_folds, "kfold")

[]